# RP-CNN without the distance channel

Rules fixed before running (recall, N=64, 4 pairs, d=32, depth 2, mean of seeds 0–2):
- **equal**: series-only R² ≥ D+series R² − 0.02 → the 2D pairwise layout does the work.
- **partial**: 0.30 ≤ series-only R² < D+series R² − 0.02 → D is a shortcut, not a requirement.
- **fail**: series-only R² < 0.30 → the distance channel is essential at this size.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from jepa_experiments.train import train_supervised
from jepa_experiments.experiment import SMOKE, get_split, cached_run, summarise

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
M, TAU, DEPTH, BATCH, LR = 3, 1, 2, 128, 1e-3
SIZES = {'train': 800, 'val': 200, 'test': 300} if SMOKE else {'train': 20000, 'val': 2000, 'test': 5000}
SEEDS = (0,) if SMOKE else (0, 1, 2)
EPOCHS, PATIENCE = (2, 1) if SMOKE else (40, 10)
D_RUN = 16 if SMOKE else 32
N_MAIN = 32 if SMOKE else 64
CACHE_ROOT = Path('cache/step2')

TABLE_ROOT = Path('tables/step6') / ('smoke' if SMOKE else '')
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
SPECS = {'rp_cnn': ('rp_cnn', {'rp_kind': 'D+series'}), 'rp_cnn_series': ('rp_cnn', {'rp_kind': 'series'}),
         'rp_cnn_D': ('rp_cnn', {'rp_kind': 'D'}), 'raw_attn': ('raw_attn', {})}
TRAIN_SIZES = (400,) if SMOKE else (2000, 5000)
OTHER_TASKS = ('forecast',) if SMOKE else ('system', 'lyapunov', 'noise', 'changepoint', 'forecast')
OTHER_SEEDS = SEEDS[:2]
print({'device': DEVICE, 'smoke': SMOKE, 'sizes': SIZES, 'seeds': SEEDS})

def run(spec, task, seed, depth=DEPTH, n_train=None):
    enc_name, enc_kw = SPECS[spec]
    sizes = SIZES | ({'train': n_train} if n_train else {})
    difficulty = {'n_pairs': 4} if task == 'recall' else {}
    tr, va, te = (get_split(task, s, sizes[s], N_MAIN, M, TAU, base_seed=seed, **difficulty) for s in ('train', 'val', 'test'))
    key = {'spec_label': spec, 'enc_name': enc_name, 'enc_kw': enc_kw, 'd': D_RUN, 'depth': depth, 'N': N_MAIN,
           'n_pairs': 4, 'task': task, 'seed': int(seed), 'epochs': EPOCHS, 'patience': PATIENCE,
           'sizes': sizes, 'batch': BATCH, 'lr': LR, 'm': M, 'tau': TAU}
    res = cached_run(CACHE_ROOT, key, lambda: train_supervised(
        enc_name, D_RUN, depth, tr, va, te, n_states=N_MAIN, m=M, tau=TAU, epochs=EPOCHS, batch=BATCH,
        lr=LR, seed=seed, patience=PATIENCE, device=DEVICE, enc_kw=enc_kw))
    print(spec, task, 'depth', depth, 'train', sizes['train'], 'seed', seed, round(res['metric'], 4),
          f"{res['train_s']:.0f}s", 'epochs', res['epochs_run'], flush=True)
    return {'spec': spec, 'task': task, 'depth': depth, 'n_train': sizes['train'], 'seed': seed,
            'metric': res['metric'], 'val_metric': res['val_metric'], 'epochs_run': res['epochs_run'],
            'train_s': res['train_s'], 'params': res['params'], 'flops': res['flops'], 'curve': res['curve']}

## Main comparison

In [ ]:
main_df = pd.DataFrame([run(spec, 'recall', s) for spec in ('rp_cnn', 'rp_cnn_series', 'rp_cnn_D') for s in SEEDS])
display(summarise(main_df, 'spec', 'metric').round(4))
display(main_df[['spec', 'seed', 'metric', 'epochs_run', 'params', 'flops']])

## Depth 3

In [ ]:
deep_df = pd.DataFrame([run('rp_cnn_series', 'recall', s, depth=3) for s in SEEDS])
display(summarise(deep_df, 'spec', 'metric').round(4))

## Training-set size

In [ ]:
size_df = pd.DataFrame([run(spec, 'recall', s, n_train=n) for n in TRAIN_SIZES for spec in ('rp_cnn', 'rp_cnn_series') for s in SEEDS])
size_df = pd.concat([size_df, main_df[main_df.spec.isin(['rp_cnn', 'rp_cnn_series'])]], ignore_index=True)
size_summary = summarise(size_df, ['spec', 'n_train'], 'metric')
display(size_summary.pivot(index='n_train', columns='spec', values='mean').round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for spec, g in size_summary.groupby('spec'):
    axes[0].errorbar(g.n_train, g['mean'], yerr=g['sd'].fillna(0), marker='o', capsize=3, label=spec)
axes[0].set(xscale='log', xlabel='Training windows', ylabel='Test R²', title='Recall versus training size')
for _, r in main_df[main_df.spec != 'rp_cnn_D'].iterrows():
    axes[1].plot(range(1, len(r.curve) + 1), r.curve, color='C0' if r.spec == 'rp_cnn' else 'C1', alpha=.7,
                 label=r.spec if r.seed == SEEDS[0] else None)
axes[1].set(xlabel='Epoch', ylabel='Validation R²', title=f'Learning curves, {SIZES["train"]} windows')
for ax in axes: ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

## Other tasks

In [ ]:
other_df = pd.DataFrame([run(spec, t, s) for t in OTHER_TASKS for spec in SPECS for s in OTHER_SEEDS])
display(summarise(other_df, ['task', 'spec'], 'metric').pivot(index='task', columns='spec', values='mean').round(4))

## Decision

In [ ]:
mean = main_df.groupby('spec').metric.mean()
full, series = mean['rp_cnn'], mean['rp_cnn_series']
verdict = 'equal' if series >= full - .02 else ('partial' if series >= .30 else 'fail')
print({'D+series': round(full, 4), 'series only': round(series, 4), 'D only': round(mean['rp_cnn_D'], 4),
       'series only, depth 3': round(deep_df.metric.mean(), 4), 'verdict': verdict})
pd.concat([main_df, deep_df, size_df, other_df], ignore_index=True).drop(columns='curve').drop_duplicates(
    ['spec', 'task', 'depth', 'n_train', 'seed']).to_csv(TABLE_ROOT / 'series_only_runs.csv', index=False)

## Findings

Full run: recall, N = 64, 4 pairs, d = 32, train/val/test 20,000/2,000/5,000, 40 epochs, patience 10; D+series and D-only rows from the Step 2 cache where available. Test R².

**Pre-registered verdict: partial.** Series-only reaches 0.933 (seeds 0.982, 0.850, 0.966) against 0.998 for D+series; D-only stays at 0.000.

1. **The 2D pairwise layout alone can learn recall.** Without any distance channel the same CNN reaches 0.93, and 0.973 (0.961–0.994) with one more block (depth 3). The distance matrix is not required for recall at this data size.
2. **D is a learning shortcut.** In the learning curves, D+series jumps to ~1.0 at epochs 5–7; series-only stays flat until epochs 17–20, then climbs, and one seed plateaus at 0.85.
3. **With less training it matters a lot.** At 5,000 windows D+series still reaches 0.970, series-only 0.000; at 2,000 both fail. This confounds data with optimizer steps (epochs are fixed) and with early stopping: series-only needs ~17 flat epochs before it learns, longer than the patience of 10, so some short runs were stopped before they could start learning.
4. **Other tasks (2 seeds):** series-only matches D+series on system (0.872 vs 0.868), noise (0.983 vs 0.984) and forecast (0.979 vs 0.982), and is better on lyapunov (0.945 vs 0.926, same as raw attention 0.946). It fails changepoint (0.000 vs 0.211), stopping at epochs 22–27, the same flat-start pattern as above, so this may be an optimisation failure rather than missing information.

Caveats: one width, one learning rate, one patience; data size and training length not separated.